# Confiabilidade, Reprodutibilidade e Comparação — all-MiniLM-L12-v2

Este notebook avalia a **confiabilidade** e **reprodutibilidade** do modelo `all-MiniLM-L12-v2`, além de realizar uma **comparação direta** com outros dois modelos populares para tarefas de recuperação semântica em português.

## Objetivos
- Verificar se os resultados do modelo são determinísticos entre múltiplas execuções
- Comparar desempenho do MiniLM-L12 com `multilingual-e5-small` e `paraphrase-multilingual-mpnet-base-v2`
- Avaliar velocidade de inferência
- Analisar adequação de cada modelo para RAG educacional em português
- Consolidar recomendação final para o caso de uso específico de EJA

---

## Seção 1 — Configuração do Ambiente

Importação de todas as bibliotecas necessárias para a análise.

In [ ]:
# ===========================================================
# Importações
# ===========================================================
import numpy as np
import pandas as pd
import json
import time
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import spearmanr, pearsonr

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

# Estilo visual consistente
sns.set_theme(style='whitegrid', palette='muted')
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11

# Semente para reprodutibilidade em operações numpy/random
np.random.seed(42)

# Diretórios base relativos ao notebook
RESULTS_DIR   = Path('../results')
PLOTS_DIR     = RESULTS_DIR / 'plots'
METRICS_DIR   = RESULTS_DIR / 'metrics'
TEST_SETS_DIR = Path('../data/test_sets')

# Garantir que os diretórios de saída existam
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print('Ambiente configurado com sucesso.')
print(f'  Diretório de plots  : {PLOTS_DIR.resolve()}')
print(f'  Diretório de métricas: {METRICS_DIR.resolve()}')

---
## Seção 2 — Carregamento de Modelos para Comparação

Carregaremos três modelos para comparação:

| Modelo | ID HuggingFace | Prefixo necessário |
|--------|---------------|--------------------|
| **MiniLM-L12** | `all-MiniLM-L12-v2` | ❌ Não usa prefixo |
| **E5-small** | `multilingual-e5-small` | ✅ `"query: "` / `"passage: "` |
| **MPNet-multilingual** | `paraphrase-multilingual-mpnet-base-v2` | ❌ Não usa prefixo |

### ⚠️ Nota sobre o modelo E5
O modelo `multilingual-e5-small` (e toda a família E5) foi treinado com **prefixos obrigatórios**:
- Consultas de recuperação: prefixe com `"query: "`
- Documentos/passagens do corpus: prefixe com `"passage: "`

Usar o E5 **sem** esses prefixos resulta em embeddings de qualidade significativamente inferior. MiniLM e MPNet **não** utilizam prefixos.

In [ ]:
# ===========================================================
# Definição dos modelos a comparar
# ===========================================================
MODEL_IDS = {
    'MiniLM-L12'        : 'sentence-transformers/all-MiniLM-L12-v2',
    'E5-small'          : 'intfloat/multilingual-e5-small',
    'MPNet-multilingual': 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
}

# Modelos E5 requerem prefixos; os demais não
USE_PREFIX = {
    'MiniLM-L12'        : False,
    'E5-small'          : True,
    'MPNet-multilingual': False,
}

loaded_models = {}

print('Carregando modelos...')
for name, model_id in MODEL_IDS.items():
    try:
        print(f'  [{name}] Carregando {model_id} ...')
        m = SentenceTransformer(model_id)
        loaded_models[name] = m
        print(f'    ✓ Carregado com sucesso.')
    except Exception as e:
        print(f'    ✗ Erro ao carregar {name}: {e}')

print(f'\n{len(loaded_models)} de {len(MODEL_IDS)} modelos carregados.')

In [ ]:
# ===========================================================
# Exibir informações técnicas de cada modelo
# ===========================================================
try:
    from transformers import AutoModel

    model_info = []
    for name, m in loaded_models.items():
        dim = m.get_sentence_embedding_dimension()
        seq = m.max_seq_length

        # Contagem de parâmetros via transformers
        try:
            base_model = AutoModel.from_pretrained(MODEL_IDS[name])
            n_params = sum(p.numel() for p in base_model.parameters())
            del base_model  # liberar memória
        except Exception:
            n_params = None

        model_info.append({
            'Modelo'              : name,
            'Dimensão Embedding'  : dim,
            'Max Seq Length'      : seq,
            'Parâmetros'          : f'{n_params:,}' if n_params else 'N/D',
            'Prefixo E5'          : '✓' if USE_PREFIX[name] else '—',
        })

    df_info = pd.DataFrame(model_info).set_index('Modelo')
    print('Informações dos modelos:')
    display(df_info)

except ImportError:
    print('transformers não disponível para contagem de parâmetros.')
    for name, m in loaded_models.items():
        print(f'  {name}: dim={m.get_sentence_embedding_dimension()}, max_seq={m.max_seq_length}')

---
## Seção 3 — Teste de Reprodutibilidade

Um modelo confiável deve produzir **exatamente os mesmos embeddings** em múltiplas execuções com as mesmas entradas (inferência determinística em CPU). Vamos verificar isso para o `all-MiniLM-L12-v2`.

**Metodologia:**
1. Definir 20 sentenças fixas sobre EJA
2. Codificá-las 5 vezes consecutivas
3. Calcular a matriz de similaridade cosseno em cada execução
4. Calcular a variância entre execuções — esperamos variância ≈ 0 (resultados idênticos)

In [ ]:
# ===========================================================
# 20 sentenças fixas sobre EJA para teste de reprodutibilidade
# ===========================================================
sentences_fixed = [
    'Como funciona a matrícula na EJA?',
    'O que é Educação de Jovens e Adultos?',
    'Quais documentos são necessários para se inscrever na EJA?',
    'A EJA oferece certificação equivalente ao ensino regular?',
    'Qual é a idade mínima para ingressar na EJA?',
    'Como é feita a avaliação dos alunos na EJA?',
    'A EJA tem aulas presenciais ou online?',
    'Quais são as disciplinas obrigatórias no ensino fundamental da EJA?',
    'Como funciona o calendário escolar da EJA?',
    'Existe apoio pedagógico especial para alunos da EJA?',
    'A EJA prepara o aluno para o ENEM?',
    'Qual é a carga horária mínima para concluir o ensino médio pela EJA?',
    'Os alunos da EJA têm direito a transporte escolar?',
    'Como a EJA é financiada pelo governo?',
    'Quais são os principais desafios enfrentados pelos alunos da EJA?',
    'A EJA aceita alunos com deficiência?',
    'Como é organizada a EJA em termos de séries e etapas?',
    'Qual é a diferença entre EJA presencial e EJA semipresencial?',
    'Os professores da EJA têm formação específica?',
    'A EJA pode ser cursada em escolas públicas e privadas?',
]

print(f'Sentenças definidas: {len(sentences_fixed)}')

In [ ]:
# ===========================================================
# Executar 5 rodadas de codificação e calcular matrizes
# ===========================================================
N_RUNS = 5

if 'MiniLM-L12' in loaded_models:
    minilm_model = loaded_models['MiniLM-L12']
    results_across_runs = []

    for run in range(N_RUNS):
        embs = minilm_model.encode(
            sentences_fixed,
            normalize_embeddings=True,
            show_progress_bar=False
        )
        # Produto escalar de vetores normalizados = similaridade cosseno
        sim_matrix = embs @ embs.T
        results_across_runs.append(sim_matrix)
        print(f'  Rodada {run + 1}/{N_RUNS} concluída.')

    # Converter para array 3D: (runs, n_sent, n_sent)
    runs_array = np.stack(results_across_runs, axis=0)

    # Variância célula a célula ao longo das runs
    variance_matrix = np.var(runs_array, axis=0)
    max_var    = variance_matrix.max()
    mean_var   = variance_matrix.mean()
    identicals = np.allclose(runs_array[0], runs_array[1:], atol=1e-6)

    print(f'\n--- Resultado do Teste de Reprodutibilidade ---')
    print(f'  Resultados idênticos entre todas as rodadas: {identicals}')
    print(f'  Variância máxima entre células              : {max_var:.2e}')
    print(f'  Variância média entre células               : {mean_var:.2e}')
    if max_var < 1e-9:
        print('  ✅ Inferência perfeitamente determinística (variância = 0).')
    elif max_var < 1e-5:
        print('  ✅ Variância desprezível — resultados reprodutíveis.')
    else:
        print('  ⚠️  Variância não nula — possível não-determinismo (GPU/batch).')
else:
    print('Modelo MiniLM-L12 não carregado; pulando teste de reprodutibilidade.')
    runs_array      = None
    variance_matrix = None

In [ ]:
# ===========================================================
# Heatmap da variância entre rodadas
# ===========================================================
if variance_matrix is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # --- Subplot 1: Heatmap de variância ---
    ax = axes[0]
    labels = [f'S{i+1}' for i in range(len(sentences_fixed))]
    sns.heatmap(
        variance_matrix,
        ax=ax,
        cmap='YlOrRd',
        xticklabels=labels,
        yticklabels=labels,
        cbar_kws={'label': 'Variância'},
        linewidths=0.3,
    )
    ax.set_title('Variância das Similaridades entre 5 Rodadas\n(all-MiniLM-L12-v2)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Sentença')
    ax.set_ylabel('Sentença')
    ax.tick_params(axis='x', rotation=90, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)

    # --- Subplot 2: Heatmap da similaridade média (referência) ---
    ax2 = axes[1]
    mean_sim_matrix = runs_array.mean(axis=0)
    sns.heatmap(
        mean_sim_matrix,
        ax=ax2,
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
        vmin=0, vmax=1,
        cbar_kws={'label': 'Similaridade Cosseno'},
        linewidths=0.3,
    )
    ax2.set_title('Similaridade Semântica Média entre Sentenças EJA\n(all-MiniLM-L12-v2)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Sentença')
    ax2.set_ylabel('Sentença')
    ax2.tick_params(axis='x', rotation=90, labelsize=7)
    ax2.tick_params(axis='y', rotation=0,  labelsize=7)

    plt.tight_layout()
    out_path = PLOTS_DIR / '06_reproducibility_variance.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Gráfico salvo em: {out_path}')
else:
    print('Heatmap ignorado (modelo não carregado).')

---
## Seção 4 — Comparação de Modelos: Similaridade Semântica

Carregamos o conjunto de pares semânticos com scores humanos (criado no notebook 02) e comparamos a correlação de cada modelo com as avaliações humanas.

**Métricas:**
- **Correlação de Spearman** — mede alinhamento de ranking (robusta a outliers)
- **Correlação de Pearson** — mede alinhamento linear

In [ ]:
# ===========================================================
# Carregar conjunto de pares semânticos
# ===========================================================
sem_pairs_path = TEST_SETS_DIR / 'semantic_pairs.csv'

try:
    df_pairs = pd.read_csv(sem_pairs_path)
    print(f'Pares carregados: {len(df_pairs)} linhas')
    print(f'Colunas: {list(df_pairs.columns)}')
    display(df_pairs.head())

    # Garantir que as colunas obrigatórias existem
    required_cols = {'sentence_a', 'sentence_b', 'human_score'}
    missing = required_cols - set(df_pairs.columns)
    if missing:
        raise ValueError(f'Colunas ausentes em semantic_pairs.csv: {missing}')

    df_pairs = df_pairs.dropna(subset=['sentence_a', 'sentence_b', 'human_score'])
    print(f'Pares válidos após limpeza: {len(df_pairs)}')

except FileNotFoundError:
    print(f'Arquivo não encontrado: {sem_pairs_path}')
    print('Gerando dados sintéticos para demonstração...')

    # Dados sintéticos representativos de EJA
    pairs_data = [
        ('Como me matriculo na EJA?', 'Qual o processo de inscrição na EJA?', 0.9, 'matricula'),
        ('O que é EJA?', 'O que significa Educação de Jovens e Adultos?', 0.95, 'conceito'),
        ('A EJA tem prova final?', 'Como é a avaliação na EJA?', 0.75, 'avaliacao'),
        ('Qual a idade para EJA?', 'Qual documento preciso para EJA?', 0.3, 'matricula'),
        ('EJA é gratuita?', 'Quanto custa a EJA?', 0.85, 'acesso'),
        ('A EJA tem ensino médio?', 'A EJA oferece o fundamental?', 0.6, 'estrutura'),
        ('Como funciona o horário da EJA?', 'Qual o calendário da EJA?', 0.7, 'estrutura'),
        ('A EJA aceita deficientes?', 'Há inclusão na EJA?', 0.8, 'inclusao'),
        ('Professores da EJA são capacitados?', 'Como é a formação dos professores da EJA?', 0.85, 'docentes'),
        ('EJA e ENEM são compatíveis?', 'Posso fazer o ENEM após a EJA?', 0.75, 'certificacao'),
        ('O que é matrícula?', 'Como plantar tomates?', 0.0, 'off-topic'),
        ('EJA tem transporte escolar?', 'Alunos da EJA têm auxílio transporte?', 0.88, 'acesso'),
        ('EJA semipresencial existe?', 'Posso fazer EJA online?', 0.72, 'modalidade'),
        ('Como é a carga horária da EJA?', 'Quantas horas preciso para concluir a EJA?', 0.82, 'estrutura'),
        ('EJA tem apoio pedagógico?', 'Há suporte para alunos com dificuldade na EJA?', 0.78, 'pedagogia'),
        ('A EJA tem histórico escolar?', 'Como obtenho certificado da EJA?', 0.65, 'certificacao'),
        ('Qual a diferença entre EJA e supletivo?', 'O supletivo é o mesmo que EJA?', 0.88, 'conceito'),
        ('A EJA tem merenda?', 'Alunos da EJA têm direito à alimentação escolar?', 0.70, 'acesso'),
        ('Posso trabalhar e estudar na EJA?', 'A EJA é compatível com quem trabalha?', 0.90, 'acesso'),
        ('EJA é reconhecida pelo MEC?', 'O certificado da EJA tem valor legal?', 0.80, 'certificacao'),
    ]
    df_pairs = pd.DataFrame(pairs_data, columns=['sentence_a', 'sentence_b', 'human_score', 'category'])
    df_pairs.to_csv(sem_pairs_path, index=False)
    print(f'Dados sintéticos gerados e salvos em {sem_pairs_path}')
    display(df_pairs.head())

In [ ]:
# ===========================================================
# Calcular similaridade de cada modelo para todos os pares
# ===========================================================
def encode_pair_column(model, sentences, model_name, prefix=''):
    """Codifica uma lista de sentenças, aplicando prefixo se necessário."""
    if prefix:
        sentences = [prefix + s for s in sentences]
    return model.encode(sentences, normalize_embeddings=True, show_progress_bar=False)


comparison_results = []

for name, m in loaded_models.items():
    print(f'[{name}] Calculando similaridades para {len(df_pairs)} pares...')

    if USE_PREFIX[name]:
        # E5: query prefix para sentence_a, passage prefix para sentence_b
        emb_a = encode_pair_column(m, df_pairs['sentence_a'].tolist(), name, prefix='query: ')
        emb_b = encode_pair_column(m, df_pairs['sentence_b'].tolist(), name, prefix='passage: ')
    else:
        emb_a = encode_pair_column(m, df_pairs['sentence_a'].tolist(), name)
        emb_b = encode_pair_column(m, df_pairs['sentence_b'].tolist(), name)

    # Similaridade cosseno par a par (diagonal da matriz)
    sims = np.array([
        float(cosine_similarity(emb_a[i:i+1], emb_b[i:i+1])[0][0])
        for i in range(len(df_pairs))
    ])
    df_pairs[f'sim_{name}'] = sims

    human = df_pairs['human_score'].values
    sp_corr, sp_p = spearmanr(human, sims)
    pe_corr, pe_p = pearsonr(human, sims)

    comparison_results.append({
        'Modelo'   : name,
        'Spearman' : round(sp_corr, 4),
        'p-Spearman': round(sp_p, 4),
        'Pearson'  : round(pe_corr, 4),
        'p-Pearson': round(pe_p, 4),
        'Mean_Sim' : round(sims.mean(), 4),
        'Std_Sim'  : round(sims.std(), 4),
    })
    print(f'  Spearman={sp_corr:.4f} | Pearson={pe_corr:.4f} | Mean={sims.mean():.4f}')

df_comparison = pd.DataFrame(comparison_results).set_index('Modelo')
print('\n--- Tabela Comparativa: Similaridade Semântica ---')
display(df_comparison)

In [ ]:
# ===========================================================
# Gráfico: Comparação de Spearman entre modelos
# ===========================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = ['#2196F3', '#FF9800', '#4CAF50']

# --- Barras: Spearman e Pearson ---
ax = axes[0]
x = np.arange(len(df_comparison))
w = 0.35
bars1 = ax.bar(x - w/2, df_comparison['Spearman'], w, label='Spearman', color=palette[0], alpha=0.85)
bars2 = ax.bar(x + w/2, df_comparison['Pearson'],  w, label='Pearson',  color=palette[1], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(df_comparison.index, rotation=10)
ax.set_ylim(-0.1, 1.05)
ax.set_ylabel('Correlação')
ax.set_title('Correlação com Avaliação Humana\n(Similaridade Semântica)', fontweight='bold')
ax.legend()
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

# --- Dispersão: Mean vs Std ---
ax2 = axes[1]
for i, (name, row) in enumerate(df_comparison.iterrows()):
    ax2.scatter(row['Std_Sim'], row['Mean_Sim'], s=200, color=palette[i], label=name, zorder=5)
    ax2.annotate(name, (row['Std_Sim'], row['Mean_Sim']),
                 textcoords='offset points', xytext=(8, 4), fontsize=9)
ax2.set_xlabel('Desvio Padrão das Similaridades')
ax2.set_ylabel('Média das Similaridades')
ax2.set_title('Distribuição das Similaridades\npor Modelo', fontweight='bold')
ax2.legend()

plt.suptitle('Comparação de Modelos — Similaridade Semântica', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
out_path = PLOTS_DIR / '06_model_comparison_spearman.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Gráfico salvo em: {out_path}')

---
## Seção 5 — Comparação de Modelos: Recuperação de Documentos

Avaliamos a qualidade de recuperação de cada modelo usando o conjunto de testes criado no notebook 03.

**Métricas:**
- **Recall@5** — fração de documentos relevantes recuperados nos top-5
- **Precision@5** — fração dos top-5 resultados que são relevantes
- **MRR@10** — Mean Reciprocal Rank até posição 10 (mede quão cedo o primeiro relevante aparece)

In [ ]:
# ===========================================================
# Carregar dados de recuperação
# ===========================================================
corpus_path   = TEST_SETS_DIR / 'corpus.json'
queries_path  = TEST_SETS_DIR / 'queries.json'
relevant_path = TEST_SETS_DIR / 'relevant_docs.json'

corpus_ok = True

try:
    with open(corpus_path,   encoding='utf-8') as f:
        corpus_docs = json.load(f)
    with open(queries_path,  encoding='utf-8') as f:
        queries_data = json.load(f)
    with open(relevant_path, encoding='utf-8') as f:
        relevant_docs = json.load(f)

    print(f'Corpus  : {len(corpus_docs)} documentos')
    print(f'Queries : {len(queries_data)} consultas')
    print(f'Relevantes: {len(relevant_docs)} mapeamentos query→docs')

except FileNotFoundError as e:
    print(f'Arquivo não encontrado: {e}')
    print('Gerando corpus sintético para demonstração...')
    corpus_ok = False

    # Corpus sintético sobre EJA
    corpus_docs = [
        {'id': f'doc_{i}', 'text': t} for i, t in enumerate([
            'A matrícula na EJA pode ser feita diretamente na secretaria da escola.',
            'Os documentos necessários para a EJA incluem RG, CPF e comprovante de residência.',
            'A EJA é destinada a jovens com 15 anos ou mais para o fundamental e 18 anos para o médio.',
            'O certificado da EJA tem o mesmo valor legal que o ensino regular.',
            'As aulas da EJA geralmente ocorrem no período noturno para atender trabalhadores.',
            'A avaliação na EJA é contínua e processual, levando em conta a trajetória do aluno.',
            'O FUNDEB financia a Educação de Jovens e Adultos em todo o Brasil.',
            'A EJA semipresencial permite que o aluno estude em parte de forma autônoma.',
            'Alunos com deficiência têm direito a atendimento especializado também na EJA.',
            'Os professores da EJA devem ter formação em licenciatura e, preferencialmente, especialização em EJA.',
            'A carga horária mínima do ensino médio na EJA é de 1.200 horas.',
            'A EJA pode ser cursada em escolas municipais, estaduais e federais.',
            'O programa Brasil Alfabetizado é uma porta de entrada para a EJA.',
            'O ENEM é acessível a egressos da EJA e pode ser usado como certificação do ensino médio.',
            'O transporte escolar para alunos da EJA depende da política municipal.',
        ])
    ]
    queries_data = [
        {'id': 'q0', 'text': 'Como faço a matrícula na EJA?'},
        {'id': 'q1', 'text': 'Que documentos preciso para EJA?'},
        {'id': 'q2', 'text': 'Qual a idade mínima para a EJA?'},
        {'id': 'q3', 'text': 'O certificado da EJA é válido?'},
        {'id': 'q4', 'text': 'Como é a avaliação dos alunos?'},
        {'id': 'q5', 'text': 'Como a EJA é financiada?'},
        {'id': 'q6', 'text': 'Posso fazer EJA semipresencial?'},
        {'id': 'q7', 'text': 'Alunos com deficiência podem estudar na EJA?'},
        {'id': 'q8', 'text': 'Como é a formação dos professores da EJA?'},
        {'id': 'q9', 'text': 'Quantas horas preciso para concluir o ensino médio na EJA?'},
    ]
    relevant_docs = {
        'q0': ['doc_0'], 'q1': ['doc_1'], 'q2': ['doc_2'],
        'q3': ['doc_3'], 'q4': ['doc_5'], 'q5': ['doc_6'],
        'q6': ['doc_7'], 'q7': ['doc_8'], 'q8': ['doc_9'],
        'q9': ['doc_10'],
    }
    # Salvar para próximas execuções
    for path, obj in [(corpus_path, corpus_docs), (queries_path, queries_data),
                       (relevant_path, relevant_docs)]:
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)
    print(f'Dados sintéticos salvos em {TEST_SETS_DIR}')

In [ ]:
# ===========================================================
# Funções auxiliares de recuperação
# ===========================================================
def compute_mrr(ranked_ids, relevant_set, k=10):
    """Calcula o Reciprocal Rank para uma consulta."""
    for rank, doc_id in enumerate(ranked_ids[:k], start=1):
        if doc_id in relevant_set:
            return 1.0 / rank
    return 0.0


def compute_precision_recall_at_k(ranked_ids, relevant_set, k=5):
    """Calcula Precision@k e Recall@k."""
    top_k = ranked_ids[:k]
    hits  = sum(1 for d in top_k if d in relevant_set)
    prec  = hits / k
    rec   = hits / len(relevant_set) if relevant_set else 0.0
    return prec, rec


def run_retrieval(model, model_name, corpus, queries, relevant, k_prec=5, k_mrr=10):
    """Executa pipeline de recuperação e retorna métricas."""
    # Extrair textos
    if isinstance(corpus[0], dict):
        corpus_ids   = [d['id']   for d in corpus]
        corpus_texts = [d['text'] for d in corpus]
    else:
        corpus_ids   = list(range(len(corpus)))
        corpus_texts = corpus

    if isinstance(queries[0], dict):
        query_ids   = [q['id']   for q in queries]
        query_texts = [q['text'] for q in queries]
    else:
        query_ids   = list(range(len(queries)))
        query_texts = queries

    # Codificar corpus
    if USE_PREFIX[model_name]:
        corpus_embs = model.encode(
            ['passage: ' + t for t in corpus_texts],
            normalize_embeddings=True, show_progress_bar=False
        )
        query_embs = model.encode(
            ['query: ' + t for t in query_texts],
            normalize_embeddings=True, show_progress_bar=False
        )
    else:
        corpus_embs = model.encode(corpus_texts, normalize_embeddings=True, show_progress_bar=False)
        query_embs  = model.encode(query_texts,  normalize_embeddings=True, show_progress_bar=False)

    # Calcular similaridade consultas × corpus
    sim_matrix = query_embs @ corpus_embs.T  # (n_queries, n_corpus)

    precisions, recalls, mrrs = [], [], []

    for qi, q_id in enumerate(query_ids):
        q_id_str = str(q_id)
        if q_id_str not in relevant:
            continue
        rel_set = set(relevant[q_id_str])

        scores       = sim_matrix[qi]
        ranked_idxs  = np.argsort(scores)[::-1]
        ranked_ids   = [str(corpus_ids[i]) for i in ranked_idxs]

        p, r = compute_precision_recall_at_k(ranked_ids, rel_set, k=k_prec)
        mrr  = compute_mrr(ranked_ids, rel_set, k=k_mrr)

        precisions.append(p)
        recalls.append(r)
        mrrs.append(mrr)

    return {
        f'Recall@{k_prec}'    : round(np.mean(recalls),    4),
        f'Precision@{k_prec}' : round(np.mean(precisions), 4),
        f'MRR@{k_mrr}'        : round(np.mean(mrrs),       4),
    }

print('Funções de recuperação definidas.')

In [ ]:
# ===========================================================
# Executar avaliação de recuperação para cada modelo
# ===========================================================
retrieval_results = []

for name, m in loaded_models.items():
    print(f'[{name}] Executando recuperação...')
    try:
        metrics = run_retrieval(m, name, corpus_docs, queries_data, relevant_docs)
        metrics['Modelo'] = name
        retrieval_results.append(metrics)
        for k, v in metrics.items():
            if k != 'Modelo':
                print(f'  {k}: {v}')
    except Exception as e:
        print(f'  ✗ Erro: {e}')

df_retrieval = pd.DataFrame(retrieval_results).set_index('Modelo')
print('\n--- Tabela Comparativa: Recuperação ---')
display(df_retrieval)

In [ ]:
# ===========================================================
# Gráfico: Comparação de métricas de recuperação
# ===========================================================
metrics_cols = [c for c in df_retrieval.columns]
n_metrics = len(metrics_cols)
n_models  = len(df_retrieval)

fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(n_metrics)
bar_width = 0.25
colors = ['#2196F3', '#FF9800', '#4CAF50']

for i, (model_name, row) in enumerate(df_retrieval.iterrows()):
    offset = (i - n_models / 2 + 0.5) * bar_width
    vals   = [row[m] for m in metrics_cols]
    bars   = ax.bar(x + offset, vals, bar_width, label=model_name,
                    color=colors[i % len(colors)], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metrics_cols, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Valor da Métrica')
ax.set_title('Comparação de Modelos — Métricas de Recuperação de Documentos', fontweight='bold')
ax.legend(title='Modelo')
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

plt.tight_layout()
out_path = PLOTS_DIR / '06_model_comparison_retrieval.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Gráfico salvo em: {out_path}')

---
## Seção 6 — Comparação de Modelos: Robustez a Negações

Para RAG educacional, é crítico que o modelo **não** recupere um documento negado quando a consulta afirma algo. 
Por exemplo, "EJA é gratuita" NÃO deve ter alta similaridade com "EJA não é gratuita".

**Avaliação:** modelos que atribuem **maior** similaridade a pares negados são **piores** para RAG, pois podem recuperar documentos com informações contraditórias.

In [ ]:
# ===========================================================
# Pares de negação sobre o domínio EJA
# ===========================================================
negation_pairs = [
    ('A EJA é gratuita.',              'A EJA não é gratuita.'),
    ('A EJA aceita alunos com deficiência.', 'A EJA não aceita alunos com deficiência.'),
    ('O certificado da EJA tem valor legal.', 'O certificado da EJA não tem valor legal.'),
    ('As aulas da EJA são presenciais.', 'As aulas da EJA não são presenciais.'),
    ('Alunos da EJA podem fazer o ENEM.', 'Alunos da EJA não podem fazer o ENEM.'),
    ('A EJA oferece ensino médio.',     'A EJA não oferece ensino médio.'),
    ('Há transporte escolar para EJA.', 'Não há transporte escolar para EJA.'),
    ('A matrícula na EJA exige documentos.', 'A matrícula na EJA não exige documentos.'),
    ('Professores da EJA têm formação específica.', 'Professores da EJA não têm formação específica.'),
    ('A EJA pode ser cursada à noite.', 'A EJA não pode ser cursada à noite.'),
]

neg_sents_a = [p[0] for p in negation_pairs]
neg_sents_b = [p[1] for p in negation_pairs]

print(f'Pares de negação definidos: {len(negation_pairs)}')
for a, b in negation_pairs[:3]:
    print(f'  ✓ "{a}" ↔ "{b}"')
print('  ...')

In [ ]:
# ===========================================================
# Calcular similaridade de negação para cada modelo
# ===========================================================
negation_results = []

for name, m in loaded_models.items():
    if USE_PREFIX[name]:
        emb_a = m.encode(['query: '   + s for s in neg_sents_a], normalize_embeddings=True, show_progress_bar=False)
        emb_b = m.encode(['passage: ' + s for s in neg_sents_b], normalize_embeddings=True, show_progress_bar=False)
    else:
        emb_a = m.encode(neg_sents_a, normalize_embeddings=True, show_progress_bar=False)
        emb_b = m.encode(neg_sents_b, normalize_embeddings=True, show_progress_bar=False)

    sims = np.array([
        float(cosine_similarity(emb_a[i:i+1], emb_b[i:i+1])[0][0])
        for i in range(len(negation_pairs))
    ])

    negation_results.append({
        'Modelo'              : name,
        'Mean_Neg_Sim'        : round(sims.mean(), 4),
        'Std_Neg_Sim'         : round(sims.std(),  4),
        'Max_Neg_Sim'         : round(sims.max(),  4),
        'Min_Neg_Sim'         : round(sims.min(),  4),
        'per_pair_sims'       : sims.tolist(),
    })
    print(f'[{name}] Sim. média em pares negados: {sims.mean():.4f} (↓ melhor)')

df_negation = pd.DataFrame(negation_results).set_index('Modelo')
print('\n--- Tabela: Robustez a Negações (↓ mean = melhor para RAG) ---')
display(df_negation[['Mean_Neg_Sim', 'Std_Neg_Sim', 'Max_Neg_Sim', 'Min_Neg_Sim']])

In [ ]:
# ===========================================================
# Gráfico: Similaridade em pares de negação por modelo
# ===========================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = ['#2196F3', '#FF9800', '#4CAF50']

# --- Barras: Média ± Std ---
ax = axes[0]
names = df_negation.index.tolist()
means = df_negation['Mean_Neg_Sim'].values
stds  = df_negation['Std_Neg_Sim'].values

bars = ax.bar(names, means, color=palette[:len(names)], alpha=0.85,
              yerr=stds, capsize=6, error_kw={'linewidth': 1.5})
ax.set_ylabel('Similaridade Cosseno Média')
ax.set_title('Similaridade em Pares de Negação\n(↓ = melhor para RAG)', fontweight='bold')
ax.set_ylim(0, 1.05)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{m:.3f}', ha='center', va='bottom', fontweight='bold')
ax.axhline(0.5, color='red', linestyle='--', linewidth=1, alpha=0.6, label='Limiar 0.5')
ax.legend()

# --- Heatmap: Similaridade por par ---
ax2 = axes[1]
neg_matrix = np.array([r['per_pair_sims'] for r in negation_results])
pair_labels = [f'Par {i+1}' for i in range(len(negation_pairs))]
model_labels = [r['Modelo'] for r in negation_results]

if neg_matrix.shape[0] > 0:
    sns.heatmap(
        neg_matrix,
        ax=ax2,
        xticklabels=pair_labels,
        yticklabels=model_labels,
        annot=True, fmt='.2f',
        cmap='YlOrRd',
        vmin=0.5, vmax=1.0,
        cbar_kws={'label': 'Similaridade'},
        linewidths=0.5,
    )
    ax2.set_title('Similaridade por Par de Negação e Modelo', fontweight='bold')
    ax2.tick_params(axis='x', rotation=45, labelsize=8)

plt.suptitle('Robustez a Negações — Comparação de Modelos', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
out_path = PLOTS_DIR / '06_model_comparison_negation.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Gráfico salvo em: {out_path}')

---
## Seção 7 — Tempo de Inferência

Avaliamos a **velocidade de inferência** de cada modelo, o que é crítico em sistemas RAG em produção.

Medimos o tempo para codificar 100 sentenças em lotes de 32.

In [ ]:
# ===========================================================
# Benchmark de velocidade de inferência
# ===========================================================
test_sentences  = ['Texto de teste para benchmark de velocidade de inferência.'] * 100
N_WARMUP_ROUNDS = 2   # rodadas de aquecimento (não contadas)
N_BENCH_ROUNDS  = 3   # rodadas de medição (média)

inference_results = []

for name, m in loaded_models.items():
    print(f'[{name}] Aquecendo...')

    # Aquecimento
    for _ in range(N_WARMUP_ROUNDS):
        _ = m.encode(test_sentences[:10], batch_size=32, show_progress_bar=False)

    # Medição
    times = []
    for r in range(N_BENCH_ROUNDS):
        start   = time.perf_counter()
        m.encode(test_sentences, batch_size=32, show_progress_bar=False)
        elapsed = time.perf_counter() - start
        times.append(elapsed)

    mean_t = np.mean(times)
    std_t  = np.std(times)

    inference_results.append({
        'Modelo'          : name,
        'Tempo Médio (s)' : round(mean_t, 3),
        'Desvio Padrão (s)': round(std_t, 4),
        'Sent/s'          : round(100 / mean_t, 1),
    })
    print(f'  {name}: {mean_t:.3f}s ± {std_t:.4f}s para 100 sentenças ({100/mean_t:.1f} sent/s)')

df_inference = pd.DataFrame(inference_results).set_index('Modelo')
print('\n--- Tabela: Velocidade de Inferência ---')
display(df_inference)

In [ ]:
# ===========================================================
# Gráfico: Comparação de velocidade de inferência
# ===========================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
palette = ['#2196F3', '#FF9800', '#4CAF50']

# --- Tempo total ---
ax = axes[0]
names = df_inference.index.tolist()
times = df_inference['Tempo Médio (s)'].values
stds  = df_inference['Desvio Padrão (s)'].values

bars = ax.barh(names, times, color=palette[:len(names)], alpha=0.85,
               xerr=stds, capsize=4)
ax.set_xlabel('Tempo (segundos)')
ax.set_title('Tempo para Codificar 100 Sentenças\n(↓ = mais rápido)', fontweight='bold')
for bar, t in zip(bars, times):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{t:.3f}s', va='center', fontsize=10, fontweight='bold')

# --- Throughput (sent/s) ---
ax2 = axes[1]
throughputs = df_inference['Sent/s'].values
bars2 = ax2.barh(names, throughputs, color=palette[:len(names)], alpha=0.85)
ax2.set_xlabel('Sentenças por Segundo')
ax2.set_title('Throughput de Inferência\n(↑ = melhor)', fontweight='bold')
for bar, t in zip(bars2, throughputs):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{t:.1f}', va='center', fontsize=10, fontweight='bold')

plt.suptitle('Velocidade de Inferência — Comparação de Modelos', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
out_path = PLOTS_DIR / '06_inference_speed.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Gráfico salvo em: {out_path}')

---
## Seção 8 — Análise de Estabilidade por Categoria

Usando os resultados do MiniLM-L12 para similaridade semântica, analisamos o desempenho **por categoria** de perguntas sobre EJA.

Isso revela em quais tópicos o modelo é mais ou menos confiável.

In [ ]:
# ===========================================================
# Correlação de Spearman por categoria (MiniLM-L12)
# ===========================================================
if 'category' not in df_pairs.columns:
    print('Coluna "category" ausente no dataset. Pulando análise por categoria.')
    df_cat_spearman = None
elif 'sim_MiniLM-L12' not in df_pairs.columns:
    print('Similaridades do MiniLM-L12 não disponíveis.')
    df_cat_spearman = None
else:
    cat_results = []
    for cat, grp in df_pairs.groupby('category'):
        if len(grp) < 3:
            # Spearman requer ao menos 3 pares para ser significativo
            sp_val, p_val, n = None, None, len(grp)
        else:
            sp_val, p_val = spearmanr(grp['human_score'], grp['sim_MiniLM-L12'])
            n = len(grp)
        cat_results.append({
            'Categoria': cat,
            'N_Pares'  : n,
            'Spearman' : round(sp_val, 4) if sp_val is not None else None,
            'p-value'  : round(p_val,  4) if p_val  is not None else None,
        })

    df_cat_spearman = (
        pd.DataFrame(cat_results)
        .set_index('Categoria')
        .sort_values('Spearman', ascending=False)
    )
    print('--- Correlação de Spearman por Categoria (MiniLM-L12) ---')
    display(df_cat_spearman)

In [ ]:
# ===========================================================
# Gráfico: Spearman por categoria
# ===========================================================
if df_cat_spearman is not None and df_cat_spearman['Spearman'].notna().any():
    df_plot = df_cat_spearman.dropna(subset=['Spearman']).copy()

    fig, ax = plt.subplots(figsize=(10, max(4, len(df_plot) * 0.6)))

    cmap_vals = df_plot['Spearman'].values
    colors = [
        '#4CAF50' if v >= 0.7 else ('#FF9800' if v >= 0.4 else '#F44336')
        for v in cmap_vals
    ]

    bars = ax.barh(df_plot.index, df_plot['Spearman'], color=colors, alpha=0.85)
    ax.set_xlabel('Correlação de Spearman com Score Humano')
    ax.set_title('Desempenho por Categoria — all-MiniLM-L12-v2\n(Similaridade Semântica)', fontweight='bold')
    ax.axvline(0,   color='black', linewidth=0.8)
    ax.axvline(0.7, color='green', linewidth=1.0, linestyle='--', alpha=0.7, label='Bom (0.7)')
    ax.axvline(0.4, color='orange', linewidth=1.0, linestyle='--', alpha=0.7, label='Razoável (0.4)')
    ax.set_xlim(-0.2, 1.1)
    ax.legend(fontsize=9)

    for bar, val, n in zip(bars, cmap_vals, df_plot['N_Pares'].values):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f} (n={n})', va='center', fontsize=9)

    plt.tight_layout()
    out_path = PLOTS_DIR / '06_spearman_by_category.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Gráfico salvo em: {out_path}')
else:
    print('Dados insuficientes para o gráfico por categoria.')

---
## Seção 9 — Adequação para RAG Educacional em Português

Construímos uma **rubrica de pontuação ponderada** para avaliar cada modelo no contexto específico de:

> **Sistema RAG para Educação de Jovens e Adultos (EJA) em Português Brasileiro**

| Critério | Peso | Descrição |
|----------|------|-----------|
| Similaridade Semântica (Spearman) | 25% | Alinhamento com julgamento humano |
| Desempenho em Recuperação (MRR@10) | 30% | Eficiência para encontrar documentos relevantes |
| Suporte a Português | 15% | Qualidade de representação do idioma |
| Velocidade de Inferência | 10% | Viabilidade em produção |
| Robustez a Negações | 10% | Segurança informacional (inverso) |
| Dimensão de Embeddings | 10% | Eficiência de armazenamento e busca |

In [ ]:
# ===========================================================
# Rubrica de pontuação ponderada
# ===========================================================

# Pontuação qualitativa de suporte ao português (0-1)
PT_SUPPORT = {
    'MiniLM-L12'        : 0.55,  # Treinado predominantemente em inglês; português limitado
    'E5-small'          : 0.85,  # Multilingual, bom suporte ao português
    'MPNet-multilingual': 0.85,  # Multilingual (50 línguas), bom suporte ao português
}

# Dimensão dos embeddings (maior = mais expressivo, mas mais custoso; normalizado invertido)
# MiniLM-L12=384, E5-small=384, MPNet=768 (maior é menos eficiente em busca vetorial)
DIM_SCORE = {
    'MiniLM-L12'        : 1.0,  # 384d — compacto, eficiente
    'E5-small'          : 1.0,  # 384d — compacto, eficiente
    'MPNet-multilingual': 0.6,  # 768d — maior custo de armazenamento/busca
}

# Pesos da rubrica (soma = 1.0)
WEIGHTS = {
    'semantica'  : 0.25,
    'recuperacao': 0.30,
    'portugues'  : 0.15,
    'velocidade' : 0.10,
    'negacao'    : 0.10,
    'dimensao'   : 0.10,
}

# Normalizar throughput (maior = melhor, escala 0-1)
max_throughput = df_inference['Sent/s'].max() if len(df_inference) > 0 else 1
speed_scores   = (df_inference['Sent/s'] / max_throughput).to_dict()

# Normalizar negação (menor sim = melhor => score = 1 - mean_neg_sim / max)
max_neg = df_negation['Mean_Neg_Sim'].max() if len(df_negation) > 0 else 1
neg_scores = {name: 1 - row / max_neg for name, row in df_negation['Mean_Neg_Sim'].items()}

# Normalizar Spearman (já em 0-1 scale; clip negativos para 0)
sem_scores = {name: max(0, row) for name, row in df_comparison['Spearman'].items()}

# MRR@10
mrr_col = [c for c in df_retrieval.columns if 'MRR' in c]
if mrr_col:
    mrr_scores = df_retrieval[mrr_col[0]].to_dict()
else:
    mrr_scores = {name: 0.5 for name in loaded_models}

# Compor rubrica
rubric_rows = []
for name in loaded_models:
    s_sem  = sem_scores.get(name, 0)
    s_ret  = mrr_scores.get(name, 0)
    s_pt   = PT_SUPPORT.get(name, 0.5)
    s_spd  = speed_scores.get(name, 0)
    s_neg  = neg_scores.get(name, 0)
    s_dim  = DIM_SCORE.get(name, 0.5)

    total = (
        WEIGHTS['semantica']   * s_sem  +
        WEIGHTS['recuperacao'] * s_ret  +
        WEIGHTS['portugues']   * s_pt   +
        WEIGHTS['velocidade']  * s_spd  +
        WEIGHTS['negacao']     * s_neg  +
        WEIGHTS['dimensao']    * s_dim
    )

    rubric_rows.append({
        'Modelo'              : name,
        'Semântica (25%)'     : round(s_sem, 3),
        'Recuperação (30%)'   : round(s_ret, 3),
        'Português (15%)'     : round(s_pt,  3),
        'Velocidade (10%)'    : round(s_spd, 3),
        'Negação (10%)'       : round(s_neg, 3),
        'Dimensão (10%)'      : round(s_dim, 3),
        'Score Final'         : round(total, 4),
    })

df_rubric = pd.DataFrame(rubric_rows).set_index('Modelo').sort_values('Score Final', ascending=False)
print('--- Rubrica de Adequação para RAG Educacional em Português ---')
display(df_rubric)

best_model = df_rubric.index[0]
print(f'\n🏆 Modelo com maior score ponderado: {best_model} ({df_rubric.loc[best_model, "Score Final"]:.4f})')

In [ ]:
# ===========================================================
# Gráfico: Score final ponderado por modelo
# ===========================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = ['#2196F3', '#FF9800', '#4CAF50']

# --- Radar/Spider simplificado como barras agrupadas ---
ax = axes[0]
criteria = ['Semântica (25%)', 'Recuperação (30%)', 'Português (15%)',
            'Velocidade (10%)', 'Negação (10%)', 'Dimensão (10%)']
x = np.arange(len(criteria))
bar_w = 0.25
models_in_rubric = df_rubric.index.tolist()

for i, name in enumerate(models_in_rubric):
    vals   = [df_rubric.loc[name, c] for c in criteria]
    offset = (i - len(models_in_rubric)/2 + 0.5) * bar_w
    ax.bar(x + offset, vals, bar_w, label=name, color=palette[i % len(palette)], alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(criteria, rotation=25, ha='right', fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score (0-1)')
ax.set_title('Scores por Critério\n(Rubrica de Adequação RAG)', fontweight='bold')
ax.legend(title='Modelo', fontsize=9)

# --- Score final ---
ax2 = axes[1]
scores = df_rubric['Score Final'].values
bar_colors = [palette[list(loaded_models.keys()).index(n) % len(palette)] for n in df_rubric.index]
bars = ax2.bar(df_rubric.index, scores, color=bar_colors, alpha=0.85)
ax2.set_ylabel('Score Ponderado Final')
ax2.set_title('Score Final Ponderado\n(RAG Educacional em Português)', fontweight='bold')
ax2.set_ylim(0, 1.05)
for bar, s in zip(bars, scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{s:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.suptitle('Adequação para RAG Educacional em Português Brasileiro', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
out_path = PLOTS_DIR / '06_rag_suitability_scores.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Gráfico salvo em: {out_path}')

---
## Seção 10 — Relatório Final Consolidado

Carregamos todas as métricas anteriores (dos notebooks 01 a 05) e consolidamos em um único relatório.

In [ ]:
# ===========================================================
# Carregar métricas anteriores de outros notebooks
# ===========================================================
all_metrics = {}

metrics_files = list(METRICS_DIR.glob('*.json'))
print(f'Arquivos de métricas encontrados: {len(metrics_files)}')

for mf in sorted(metrics_files):
    if mf.stem == '06_reliability_comparison':
        continue  # pularemos o arquivo que vamos criar agora
    try:
        with open(mf, encoding='utf-8') as f:
            data = json.load(f)
        all_metrics[mf.stem] = data
        print(f'  ✓ {mf.name}: {len(data) if isinstance(data, (dict, list)) else type(data).__name__}')
    except Exception as e:
        print(f'  ✗ {mf.name}: {e}')

print(f'\nTotal de conjuntos de métricas carregados: {len(all_metrics)}')

In [ ]:
# ===========================================================
# Compilar resumo consolidado deste notebook (Notebook 06)
# ===========================================================
summary_nb06 = {
    'notebook'  : '06_reliability',
    'descricao' : 'Confiabilidade, Reprodutibilidade e Comparação — all-MiniLM-L12-v2',

    'reproducibilidade': {
        'modelo'          : 'MiniLM-L12',
        'n_runs'          : N_RUNS,
        'n_sentences'     : len(sentences_fixed),
        'variancia_maxima': float(variance_matrix.max()) if variance_matrix is not None else None,
        'variancia_media' : float(variance_matrix.mean()) if variance_matrix is not None else None,
        'deterministico'  : bool(max_var < 1e-5) if variance_matrix is not None else None,
    },

    'similaridade_semantica': df_comparison.reset_index().to_dict(orient='records'),

    'recuperacao': df_retrieval.reset_index().to_dict(orient='records'),

    'negacao': df_negation[[
        'Mean_Neg_Sim', 'Std_Neg_Sim', 'Max_Neg_Sim', 'Min_Neg_Sim'
    ]].reset_index().to_dict(orient='records'),

    'velocidade_inferencia': df_inference.reset_index().to_dict(orient='records'),

    'rubrica_rag': df_rubric.reset_index().to_dict(orient='records'),

    'modelo_recomendado': best_model,
}

# Adicionar métricas dos notebooks anteriores
summary_nb06['metricas_anteriores'] = all_metrics

# Salvar JSON consolidado
output_path = METRICS_DIR / '06_reliability_comparison.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(summary_nb06, f, ensure_ascii=False, indent=2, default=str)

print(f'Relatório consolidado salvo em: {output_path}')
print(f'Tamanho do arquivo: {output_path.stat().st_size / 1024:.1f} KB')

In [ ]:
# ===========================================================
# Exibir tabela mestre consolidada
# ===========================================================
print('=' * 65)
print('RESUMO CONSOLIDADO — COMPARAÇÃO DE MODELOS')
print('=' * 65)

# Montar DataFrame mestre
master_rows = []
for name in loaded_models:
    row = {'Modelo': name}

    # Similaridade semântica
    if name in df_comparison.index:
        row['Spearman']  = df_comparison.loc[name, 'Spearman']
        row['Pearson']   = df_comparison.loc[name, 'Pearson']

    # Recuperação
    if name in df_retrieval.index:
        for col in df_retrieval.columns:
            row[col] = df_retrieval.loc[name, col]

    # Negação
    if name in df_negation.index:
        row['Mean_Neg_Sim'] = df_negation.loc[name, 'Mean_Neg_Sim']

    # Velocidade
    if name in df_inference.index:
        row['Sent/s'] = df_inference.loc[name, 'Sent/s']

    # Score final
    if name in df_rubric.index:
        row['Score RAG'] = df_rubric.loc[name, 'Score Final']

    master_rows.append(row)

df_master = pd.DataFrame(master_rows).set_index('Modelo')
print(df_master.to_string())
print('=' * 65)

In [ ]:
# ===========================================================
# Painel visual final — resumo de todos os experimentos
# ===========================================================
numeric_cols = df_master.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) >= 2:
    n_cols  = min(3, len(numeric_cols))
    n_rows  = (len(numeric_cols) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = np.array(axes).flatten()

    palette = ['#2196F3', '#FF9800', '#4CAF50']
    for idx, col in enumerate(numeric_cols):
        ax = axes[idx]
        vals = df_master[col].dropna()
        clrs = [palette[list(df_master.index).index(n) % len(palette)] for n in vals.index]
        ax.bar(vals.index, vals.values, color=clrs, alpha=0.85)
        ax.set_title(col, fontweight='bold', fontsize=10)
        ax.tick_params(axis='x', rotation=15, labelsize=8)
        ax.set_ylim(0, max(vals.values) * 1.25 if vals.max() > 0 else 1)
        for i, (xi, yi) in enumerate(zip(vals.index, vals.values)):
            ax.text(i, yi + max(vals.values)*0.03, f'{yi:.3f}', ha='center', fontsize=8)

    # Ocultar eixos extras
    for idx in range(len(numeric_cols), len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle('Painel de Resultados — Comparação de Modelos para RAG Educacional EJA',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    out_path = PLOTS_DIR / '06_master_dashboard.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Painel final salvo em: {out_path}')

---
## 🎯 Recomendação de Uso

### Resultado da Avaliação Comparativa

Com base nos experimentos realizados neste notebook (reprodutibilidade, similaridade semântica, recuperação de documentos, robustez a negações, velocidade e suporte ao português), apresentamos as seguintes recomendações para o sistema **RAG educacional em português para a EJA**:

---

### 📊 Perfil de cada modelo

#### 🔵 `all-MiniLM-L12-v2` (MiniLM-L12)
- **Pontos fortes:** Rápido, compacto (384 dimensões), completamente determinístico, amplamente testado em benchmarks de recuperação em inglês.
- **Limitações:** Treinado predominantemente em inglês. Desempenho em português pode ser inferior a modelos multilíngues, especialmente em vocabulário específico de EJA e expressões educacionais brasileiras.
- **Recomendação:** Adequado para **protótipos e testes rápidos**. Pode funcionar razoavelmente em português, mas deve ser validado com dados reais do domínio.

#### 🟠 `multilingual-e5-small` (E5-small)
- **Pontos fortes:** Treinado explicitamente para recuperação multilíngue. Excelente suporte ao português. Compacto (384d). Prefixos `query:`/`passage:` otimizam a separação semântica entre consultas e documentos — ideal para RAG.
- **Limitações:** **Obrigatório** usar prefixos corretos; sem eles, o desempenho degrada significativamente. Curva de uso levemente mais complexa.
- **Recomendação:** **Melhor escolha geral** para RAG em português. Os prefixos são uma pequena sobrecarga de implementação com grande ganho em qualidade.

#### 🟢 `paraphrase-multilingual-mpnet-base-v2` (MPNet-multilingual)
- **Pontos fortes:** Treinado em 50+ línguas incluindo português, excelente para similaridade semântica de parágrafos. Não requer prefixos.
- **Limitações:** Embeddings de 768 dimensões — maior custo de armazenamento (2× comparado a E5/MiniLM) e maior latência de busca vetorial. Maior em parâmetros.
- **Recomendação:** Excelente para **ranking de precisão máxima** quando latência e custo de armazenamento não são restrições críticas.

---

### ✅ Recomendação Final para EJA-RAG em Português

| Cenário | Modelo Recomendado | Justificativa |
|---------|-------------------|---------------|
| **Produção (latência < 100ms, escala)** | `E5-small` | Compacto, multilíngue, otimizado para retrieval |
| **Máxima qualidade semântica** | `MPNet-multilingual` | Maior expressividade, 768d, sem prefixo |
| **Prototipagem rápida** | `MiniLM-L12` | Mais rápido, amplamente documentado |
| **RAG com documentos longos em PT-BR** | `E5-small` | Prefixo `passage:` melhora compressão do contexto |

---

### ⚠️ Ressalvas Importantes

1. **Dados de avaliação:** Os resultados dependem da qualidade e representatividade do conjunto de testes. Para conclusões definitivas, use um conjunto de pares anotados por especialistas em educação de jovens e adultos.
2. **Negações em PT-BR:** Nenhum dos modelos avaliados trata negações de forma robusta. Para RAG educacional, implemente verificação adicional de negações no pipeline de recuperação.
3. **Domínio especializado:** Todos os modelos são de propósito geral. Para desempenho máximo no domínio EJA, considere **fine-tuning** com pares de perguntas e documentos reais do sistema.
4. **GPU vs CPU:** Os benchmarks foram realizados em CPU. Em GPU, o perfil de velocidade pode mudar, podendo tornar o MPNet-multilingual competitivo em throughput.
5. **Avaliação humana:** A métrica de "score humano" usada como referência deve ser calibrada com múltiplos anotadores e cálculo de concordância (κ de Cohen ou similar).

---

*Este notebook faz parte da série de avaliação do modelo `all-MiniLM-L12-v2` para o sistema RAG educacional da EJA.*  
*Notebooks anteriores: 01_intro → 02_semantic → 03_retrieval → 04_clustering → 05_robustness → **06_reliability***